In [0]:
!pip install openai --upgrade
!pip install textstat

In [0]:
# Databricks Notebook: 04_GUEST_EMAIL_GENERATION_EVAL_DELTA

# ==============================
# Step 1: Imports & Config
# ==============================
import time, uuid
import pandas as pd
import numpy as np
from textstat import flesch_kincaid_grade
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

client = OpenAI(api_key="")    # Requires OPENAI_API_KEY in secrets/environment



RUN_ID = str(uuid.uuid4())
MODEL_NAME = "gpt-3.5-turbo"
TEMPERATURE = 0.3

print("RUN_ID:", RUN_ID)



In [0]:
# ==============================
# Step 2: Load Source Data from Delta
# ==============================
import pyspark.sql.functions as F

scenarios_spark = (
    spark.read.table("bits_pilani.gold_sch.guest_persona_campaign_enriched")
    .orderBy(F.rand())      # randomize
    .limit(20)             # pick 200
)
scenarios = scenarios_spark.toPandas()
print("Loaded", len(scenarios), "rows from guest_persona_campaign_enriched")

# ==============================
# Step 3: Campaign Dictionary
# ==============================
campaign_dict = {
    "loyalty_reward": {
        "campaign_type": "Loyalty Reward",
        "offer_template": "Exclusive reward: $20 off your next {top_category} purchase",
        "tone": "Warm and appreciative",
        "call_to_action": "Claim Your Reward"
    },
    "reactivation": {
        "campaign_type": "We Miss You",
        "offer_template": "Here’s 25% off to welcome you back to our {top_category} collection!",
        "tone": "Friendly and encouraging",
        "call_to_action": "Come Back Today"
    },
    "seasonal_sale": {
        "campaign_type": "Seasonal Sale",
        "offer_template": "Seasonal offer: 30% off all {top_category} items!",
        "tone": "Exciting and persuasive",
        "call_to_action": "Shop the Sale"
    },
    "new_arrival": {
        "campaign_type": "New Arrival",
        "offer_template": "Discover the latest in {top_category} — just arrived!",
        "tone": "Fresh and trendy",
        "call_to_action": "Explore Now"
    }
}

# ==============================
# Step 4: Prompt Versions
# ==============================
PROMPT_VERSIONS = {
    "S1": {
        "system": "You are an email copywriter for a premium athleisure brand. Keep copy concise, human, and brand-safe.",
        "user_template": (
            "Write a short marketing email.\n"
            "- Persona: {persona_sentence}.\n"
            "- Campaign: {campaign_type}.\n"
            "- Offer: {offer}.\n"
            "- Guide rails: Tone is {tone}. End with CTA: \"{call_to_action}\". Avoid emojis."
        )
    },
    "S2": {
        "system": (
            "You are an email copywriter for a premium athleisure brand.\n"
            "Requirements:\n"
            "- Address the guest by name if available.\n"
            "- Explicitly reference persona traits and the top category.\n"
            "- Incorporate engagement features: spend, transactions, and recency.\n"
            "- Body 120–200 words, exactly one CTA, unsubscribe footer.\n"
            "- Tone: brand-safe, minimalistic, positive."
        ),
        "user_template": (
            "Write a hyper-personalized email.\n"
            "- Persona: {final_persona_sentence}.\n"
            "- Segment: {persona_cluster_label}.\n"
            "- Campaign: {campaign_type}.\n"
            "- Offer: {offer}.\n"
            "- Total spend: ${total_spend_usd:.2f}, Transactions: {total_transactions}, Days since last txn: {days_since_last_txn}.\n"
            "- Tone: {tone}.\n"
            "- Call-to-action: \"{call_to_action}\".\n"
            "- Top category: {top_category}."
        )
    }
}

def build_prompt(row, version="S1"):
    template = campaign_dict[row["campaign_key"]]
    offer = template["offer_template"].format(top_category=row["top_category"])
    pv = PROMPT_VERSIONS[version]
    system_msg = pv["system"]
    user_msg = pv["user_template"].format(
        persona_sentence=row.get("persona_sentence",""),
        final_persona_sentence=row.get("final_persona_sentence",""),
        persona_cluster_label=row.get("persona_cluster_label",""),
        campaign_type=template["campaign_type"],
        offer=offer,
        tone=template["tone"],
        call_to_action=template["call_to_action"],
        top_category=row.get("top_category",""),
        total_spend_usd=row.get("total_spend_usd",0),
        total_transactions=row.get("total_transactions",0),
        days_since_last_txn=row.get("days_since_last_txn",0)
    )
    return system_msg, user_msg



In [0]:
# ==============================
# Step 5: Email Generation
# ==============================
def generate_email_from_row(row, version="S1", model_name=MODEL_NAME, temperature=TEMPERATURE):
    system_msg, user_msg = build_prompt(row, version=version)
    full_prompt = f"[System]\n{system_msg}\n\n[User]\n{user_msg}"  # store exactly what was sent
    t0 = time.time()
    resp = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ],
        temperature=temperature
    )
    latency_ms = int((time.time() - t0) * 1000)
    text = resp.choices[0].message.content.strip()
    lines = [ln for ln in text.splitlines() if ln.strip()]
    if len(lines) == 0:
        subject, body = "[No Subject]", ""
    elif len(lines) == 1:
        subject, body = lines[0], ""
    else:
        subject, body = lines[0], "\n".join(lines[1:])
    usage = getattr(resp, "usage", None)
    in_toks = getattr(usage, "prompt_tokens", None) if usage else None
    out_toks = getattr(usage, "completion_tokens", None) if usage else None
    return subject, body, resp.id, in_toks, out_toks, latency_ms, full_prompt

# ==============================
# Step 6: Metric Helpers
# ==============================
def evaluate_constraints(email_body, row):
    checks = {
        "has_cta": any(phrase in email_body.lower() for phrase in ["shop now","buy today","explore","learn more"]),
        "word_range": 120 <= len(email_body.split()) <= 200,
        "footer": "unsubscribe" in email_body.lower()
    }
    adherence_pct = 100.0 * sum(1 for v in checks.values() if v) / len(checks)
    return checks, adherence_pct

def personalization_depth(email_body, row):
    attrs = []
    if row.get("persona_cluster_label"): attrs.append(row["persona_cluster_label"])
    if row.get("top_category"): attrs.append(row["top_category"])
    if row.get("preferred_channel"): attrs.append(row["preferred_channel"])
    used = sum(1 for a in attrs if str(a).lower() in email_body.lower())
    return used / len(attrs) if attrs else 0.0

def relevance_score(email_body, context_summary):
    e1 = client.embeddings.create(model="text-embedding-ada-002", input=email_body).data[0].embedding
    e2 = client.embeddings.create(model="text-embedding-ada-002", input=context_summary).data[0].embedding
    return float(cosine_similarity([e1], [e2])[0][0])

def readability_metrics(email_body):
    words = email_body.split()
    repetition_ratio = 1 - (len(set(words)) / max(1, len(words)))
    return {
        "fk_grade": float(flesch_kincaid_grade(email_body)),
        "repetition_ratio": float(repetition_ratio)
    }

# ==============================
# Step 7: Evaluation Runner
# ==============================
def run_evaluation_from_df(scenarios_df, prompt_version="S1"):
    records = []
    for _, row in scenarios_df.iterrows():
        subject, body, resp_id, in_tok, out_tok, latency_ms, full_prompt = generate_email_from_row(row, version=prompt_version)
        checks, adherence_pct = evaluate_constraints(body, row)
        pdi = personalization_depth(body, row)
        context_summary = f"{row.get('persona_sentence','')} | top_category={row.get('top_category','')}"
        rel = relevance_score(body, context_summary)
        read = readability_metrics(body)
        records.append({
            "run_id": RUN_ID,
            "context_id": row["MASTER_GUEST_ID"],
            "prompt_version": prompt_version,
            "prompt_text": full_prompt,                 # new column
            "subject": subject,
            "body": body,
            "word_count": len(body.split()),
            "constraint_adherence": adherence_pct,
            "pdi": pdi,
            "ctx_similarity": rel,
            "fk_grade": read["fk_grade"],
            "repetition_ratio": read["repetition_ratio"],
            "openai_response_id": resp_id,
            "input_tokens": in_tok,
            "output_tokens": out_tok,
            "latency_ms": latency_ms
        })
    return pd.DataFrame(records)



In [0]:
# ==============================
# Step 8: Run & Save Results to Delta
# ==============================
df_eval_s1 = run_evaluation_from_df(scenarios, prompt_version="S1")
df_eval_s2 = run_evaluation_from_df(scenarios, prompt_version="S2")

df_all = pd.concat([df_eval_s1, df_eval_s2], ignore_index=True)
eval_spark_df = spark.createDataFrame(df_all)

(
    eval_spark_df
    .write
    .format("delta")
    .mode("append")   # use overwrite in dev if needed
    .saveAsTable("bits_pilani.gold_sch.guest_email_generation_eval")
)

print("✅ Results for 20 sampled rows written to: bits_pilani.gold_sch.email_generation_eval")


In [0]:
df_eval = spark.read.table("bits_pilani.gold_sch.guest_email_generation_eval")
df_eval.display()

In [0]:
metrics = ["word_count", "constraint_adherence", "pdi", "ctx_similarity", 
           "fk_grade", "repetition_ratio", "latency_ms"]

summary = (df_eval.groupBy("prompt_version")
           .agg(*[F.avg(m).alias(f"{m}_avg") for m in metrics],
                *[F.expr(f"percentile_approx({m}, 0.5)").alias(f"{m}_median") for m in metrics])
          )
summary.display()


In [0]:
pivoted = (df_eval.groupBy("prompt_version")
           .agg(*[F.avg(m).alias(m) for m in metrics])
           .toPandas()
           .set_index("prompt_version")
           .T
)
pivoted["delta (S2-S1)"] = pivoted["S2"] - pivoted["S1"]
pivoted


## Interpretation of Results

- **Word Count (+51.8)** → S2 emails are significantly longer, aligning better with the **120–200 word target**.  
- **Constraint Adherence (+60.0)** → Major improvement; S2 emails respected campaign rules (CTA, footer, word length) far more than S1.  
- **PDI (0.00)** → No measurable difference in personalization depth between S1 and S2 in this run.  
- **Context Similarity (+0.029)** → S2 emails are slightly more semantically aligned with persona and campaign context.  
- **FK Grade (+1.14)** → S2 emails are somewhat more complex, but still within a broadly readable range.  
- **Repetition Ratio (+0.050)** → S2 is a bit more repetitive than S1, though still acceptable given better adherence and context.  
- **Latency (+721 ms)** → S2 responses are ~0.7s slower on average, an acceptable tradeoff given the large improvements in quality and compliance.  
 


In [0]:
import pandas as pd
from scipy.stats import mannwhitneyu

df_pd = df_eval.toPandas()

def compare_metric(metric):
    s1 = df_pd[df_pd["prompt_version"]=="S1"][metric].dropna()
    s2 = df_pd[df_pd["prompt_version"]=="S2"][metric].dropna()
    stat, p = mannwhitneyu(s1, s2, alternative="two-sided")
    return {"metric": metric, "S1_mean": s1.mean(), "S2_mean": s2.mean(), "p_value": p}

results = pd.DataFrame([compare_metric(m) for m in metrics])
results


Statistical testing using the Mann–Whitney U test confirmed that the observed improvements with S2 are highly significant for most metrics. Word count (p < 1e-7), constraint adherence (p < 1e-7), and context similarity (p < 1e-4) all showed strong evidence of improvement, validating that S2 reliably generates longer, rule-compliant, and more contextually aligned emails. Readability (FK grade, p < 0.01) and repetition ratio (p < 1e-4) differences were also statistically significant, though they indicate a modest increase in complexity and repetitiveness. Latency was significantly higher for S2 (p < 1e-5), reflecting the tradeoff in response speed. Only personalization depth index (PDI) showed no measurable difference (p = 1.0). Overall, these results reinforce that prompt optimization in S2 yields materially better email quality, with statistically robust gains in compliance and contextual relevance, at the cost of slightly longer generation times.
